# DEE v6 — SIM 22: Sweep adiabático L con N fijo  →  w_top(g₄) cosmológico**Autor:** Juan Pablo Bruschi (2026)**Versión:** v6 SIM 22 (complemento de SIM 21)**Estado:** test cosmológicamente correcto del régimen anharmónico topológico---## Pregunta científicaSIM 21 mostró que en el régimen anharmónico el exponente β del sweep en N (a L fija) deja de ser universal (β = +0,98 armónico → +1,11 a g₄=5000). Esto confirma que **la universalidad armónica se rompe**, pero el sweep N **no es directamente equivalente a la expansión cosmológica**: cambia la densidad de muestreo, no la escala física del cristal.**SIM 22 hace el sweep correcto**: N fijo (mismos modos cuánticos) y L variable (estiramiento adiabático). Esto es exactamente la expansión cósmica desde el punto de vista del sustrato: los mismos grados de libertad cuánticos sobre un volumen que crece.## Relación con w cosmológicoBajo la transformación L → L·(1+ε) manteniendo N, las distancias escalan como d_ij → d_ij·(1+ε), los pesos w_ij = 1/d_ij² → w_ij/(1+ε)², los autovalores λ_n → λ_n/(1+ε)², las frecuencias ω_n → ω_n/(1+ε), y E_zp → E_zp/(1+ε). En régimen armónico esto da exactamente la dependencia funcional E_zp ∝ L^(-1) ∝ V^(-1/3) por el teorema de simetría de escala.**En el régimen anharmónico**, la corrección Hartree 3·g₄·⟨φ²⟩ rompe esta simetría: ⟨φ²⟩ no escala trivialmente con L porque la convergencia autoconsistente acopla los modos infrarrojos con distinta sensibilidad a g₄.Si ΔE_top(L, g₄) ∝ L^α(g₄), el parámetro de ecuación de estado cosmológico extraído por virial cuántico es:**w_top(g₄) = −α(g₄) / 3**Esto es directamente comparable con la predicción central del Cap. 6 §6.4 (w_DE = −0,70 ± 0,05) y con los datos DESI/Pantheon+/CMB del Cap. 7.## Predicciones de referencia para w_top| α (sweep L) | w_top = −α/3 | Régimen ||---|---|---|| α = −1 | +1/3 ≈ +0,33 | Radiación (escala armónica clásica) || α = 0 | 0 | Polvo / CDM || α = +1 | −1/3 | Frontera de aceleración || α = +2 | **−2/3** | **Frontera holográfica** || **α = +2,1** | **−0,70** | **Predicción Cap. 6 §6.4** || α = +3 | −1 | ΛCDM exacta |Si SIM 22 da α(g₄=0) ≈ −1 → w_top armónico = +1/3 (esperable por teorema de escala), y α(g₄=5000) ≠ −1 → confirmación cuantitativa de la ruptura de universalidad con valor cosmológico específico.## Setup del test- N = 1331 fijo (N_side = 11) — sweet spot velocidad/precisión- L_VALS = [0.70, 0.85, 1.00, 1.20, 1.40] — rango 2× para ajuste log-log robusto- G4_VALS = [0, 100, 1000, 5000] — descarta 10⁴ (fuera de Hartree en SIM 21)- Topologías T³ y R³ sobre los mismos nodos (mismo método que SIM 21)- Hartree autoconsistente con indicador χ_Hartree- Observable: ΔE(L, g₄) = E_T³(L, g₄) − E_R³(L, g₄)- Extracción de α(g₄) por ajuste log-log de |ΔE| vs L## Tiempo estimado5 L × 4 g₄ × 2 topologías × Hartree iter (5-8). Diagonalización 1331×1331 toma ~2 s, Hartree ~15 s por punto. Total: **25–35 min** en Colab CPU.

---## 1. Setup

In [ ]:
import os, time, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

OUT_DIR = '/content/dee_v6_sweepL'
os.makedirs(OUT_DIR, exist_ok=True)

def out_path(name): return os.path.join(OUT_DIR, name)

print(f'Outputs:  {OUT_DIR}')
print(f'NumPy:    {np.__version__}')

In [ ]:
# Paleta y estilo consistente
BG  = '#0d1117'
CW  = '#ecf0f1'
CY  = '#f1c40f'
CG  = '#27ae60'
CB  = '#2980b9'
CR  = '#e74c3c'
CO  = '#e67e22'
CGR = '#7f8c8d'
CP  = '#9b59b6'

def setup_dark_axes(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=CGR)
    for s in ax.spines.values():
        s.set_color('#2c3e50')

def new_figure(nrows, ncols, figsize):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    fig.patch.set_facecolor('#0a0a1a')
    if hasattr(axes, 'flat'):
        for ax in axes.flat: setup_dark_axes(ax)
    else:
        setup_dark_axes(axes)
    return fig, axes

---## 2. Configuración globalPara test rápido: reducí `L_VALS` a 3 puntos o `G4_VALS` a 3. Total de puntos = len(L_VALS) × len(G4_VALS).

In [ ]:
# ───── parámetros ─────
N_SIDE       = 11           # → N = 1331
L_VALS       = [0.70, 0.85, 1.00, 1.20, 1.40]  # rango 2× para ajuste log-log
G4_VALS      = [0.0, 100.0, 1000.0, 5000.0]    # descartamos 10⁴ (fuera Hartree)

K_NEIGHBORS  = 30
JITTER       = 0.05
EPS_REG      = 1e-6
HARTREE_MAX  = 25
HARTREE_TOL  = 1e-5

print(f'N        = {N_SIDE**3}  (N_side = {N_SIDE})')
print(f'L_VALS   = {L_VALS}')
print(f'G4_VALS  = {G4_VALS}')
print(f'Total puntos: {len(L_VALS) * len(G4_VALS) * 2} (L × g₄ × topologías)')

---## 3. Funciones core**Diferencia clave con SIM 21:** acá la red se genera **una sola vez** en caja unitaria, y luego se reescala uniformemente por L. Esto preserva exactamente la geometría relativa entre nodos — solo cambia la escala absoluta, que es precisamente la expansión adiabática.

In [ ]:
def build_jittered_lattice_base(N_side, jitter=JITTER, seed=42):
    """Red base en caja [0,1]³ — se escalará por L después."""
    np.random.seed(seed)
    coords = np.zeros((N_side**3, 3))
    spacing = 1.0 / N_side
    idx = 0
    for i in range(N_side):
        for j in range(N_side):
            for k in range(N_side):
                base = np.array([i, j, k]) * spacing + spacing/2
                jit = spacing * jitter * np.random.uniform(-1, 1, 3)
                coords[idx] = base + jit
                idx += 1
    return coords

def pbc_distance_matrix(coords, L):
    """Distancias periódicas en caja [0, L]³."""
    N = len(coords)
    D2 = np.zeros((N, N))
    for dim in range(3):
        d1 = np.abs(coords[:, dim:dim+1] - coords[:, dim:dim+1].T)
        d1 = np.minimum(d1, L - d1)
        D2 += d1**2
    D = np.sqrt(D2)
    np.fill_diagonal(D, np.inf)
    return D

def open_distance_matrix(coords):
    """Distancias euclídeas estándar."""
    D = cdist(coords, coords)
    np.fill_diagonal(D, np.inf)
    return D

def build_laplacian(D, k_neighbors=K_NEIGHBORS):
    """Laplaciano con k-NN y kernel 1/d²."""
    N = D.shape[0]
    W = np.zeros((N, N))
    for i in range(N):
        idx = np.argsort(D[i])[:k_neighbors]
        for j in idx:
            if D[i, j] > 0 and np.isfinite(D[i, j]):
                W[i, j] = 1.0 / D[i, j]**2
    W = np.maximum(W, W.T)
    return np.diag(W.sum(axis=1)) - W

def hartree_self_consistent(K_base, g4, max_iter=HARTREE_MAX, tol=HARTREE_TOL,
                             eps_reg=EPS_REG):
    """Hartree autoconsistente con K_eff = K_base + 3·g₄·diag(⟨φ²⟩)."""
    N = K_base.shape[0]
    K_reg = K_base + eps_reg * np.eye(N)
    K_eff = K_reg.copy()
    phi_sq_prev = None
    chi_h = 0.0

    for it in range(max_iter):
        lam, vec = np.linalg.eigh(K_eff)
        lam = np.maximum(lam, eps_reg)
        inv_sqrt_diag = (vec**2 @ (1.0 / np.sqrt(lam)))
        phi_sq = 0.5 * inv_sqrt_diag

        correction_norm = np.linalg.norm(3.0 * g4 * phi_sq)
        K_base_norm = np.linalg.norm(K_base)
        chi_h = correction_norm / max(K_base_norm, 1e-10)

        K_eff_new = K_reg + 3.0 * g4 * np.diag(phi_sq)

        if phi_sq_prev is not None:
            change = np.max(np.abs(phi_sq - phi_sq_prev) / (np.abs(phi_sq_prev) + 1e-10))
            if change < tol:
                break

        K_eff = K_eff_new
        phi_sq_prev = phi_sq.copy()

    lam_final = np.linalg.eigvalsh(K_eff)
    lam_final = np.maximum(lam_final, eps_reg)
    return K_eff, lam_final, chi_h, it + 1

def E_zp_from_lambda(lam):
    return 0.5 * np.sum(np.sqrt(np.maximum(lam, 0.0)))

---## 4. Función principal: medir ΔE_top(L, g₄) con N fijoLa red base se genera **una sola vez** y se reusa escalada por cada L. Esto garantiza que solo cambia la escala absoluta — la geometría relativa, los vecinos, la conectividad son idénticos. Es la expansión adiabática estricta.

In [ ]:
# Generar red base una sola vez
print(f'Generando red base con N = {N_SIDE**3} nodos...')
coords_base = build_jittered_lattice_base(N_SIDE)
print(f'  shape: {coords_base.shape}')

def medir_punto_L(L_val, g4, verbose=False):
    """Mide E_zp_T³, E_zp_R³, ΔE para caja [0, L]³ y g₄ dado."""
    coords_scaled = coords_base * L_val

    # T³
    D_T3 = pbc_distance_matrix(coords_scaled, L=L_val)
    L_T3 = build_laplacian(D_T3)
    _, lam_T3, chi_T3, it_T3 = hartree_self_consistent(L_T3, g4)
    E_T3 = E_zp_from_lambda(lam_T3)

    # R³
    D_R3 = open_distance_matrix(coords_scaled)
    L_R3 = build_laplacian(D_R3)
    _, lam_R3, chi_R3, it_R3 = hartree_self_consistent(L_R3, g4)
    E_R3 = E_zp_from_lambda(lam_R3)

    dE = E_T3 - E_R3

    if verbose:
        print(f'  L={L_val:.2f}, g₄={g4:>5.0f}:  E_T³={E_T3:.2f}  E_R³={E_R3:.2f}  ΔE={dE:.2f}')
        print(f'    χ_H(T³)={chi_T3:.4f}  χ_H(R³)={chi_R3:.4f}  iter T³={it_T3}  R³={it_R3}')

    return dict(L=L_val, g4=g4, V=L_val**3,
                E_T3=E_T3, E_R3=E_R3, dE=dE,
                chi_T3=chi_T3, chi_R3=chi_R3,
                iter_T3=it_T3, iter_R3=it_R3)

---## 5. Barrido principal20 mediciones (5 L × 4 g₄). Persistencia en JSON: si Colab se desconecta, retoma desde donde quedó.

In [ ]:
SIM22_FILE = out_path('sim22_progress.json')

if os.path.exists(SIM22_FILE):
    with open(SIM22_FILE) as f:
        results = json.load(f)
    print(f'Progreso previo: {len(results)} puntos cargados')
else:
    results = []

ya_hechos = set((r['L'], r['g4']) for r in results)
faltantes = [(L, g4) for L in L_VALS for g4 in G4_VALS
             if (L, g4) not in ya_hechos]
total = len(L_VALS) * len(G4_VALS)
print(f'Puntos totales: {total}, faltantes: {len(faltantes)}')

t_start = time.time()
for k, (L_val, g4) in enumerate(faltantes):
    print(f'\n[{k+1}/{len(faltantes)}] L={L_val:.2f}, g₄={g4:.0f} ...')
    t0 = time.time()
    pt = medir_punto_L(L_val, g4, verbose=True)
    dt = time.time() - t0
    print(f'    → {dt:.1f}s')
    results.append(pt)
    with open(SIM22_FILE, 'w') as f:
        json.dump(results, f, indent=1)

print(f'\nTotal: {len(results)} puntos en {time.time()-t_start:.1f}s')

---## 6. Análisis: α(g₄), w_top(g₄) y comparación con SIM 21Para cada g₄ ajustamos |ΔE| ∝ L^α (sweep adiabático correcto), y de ahí extraemos w_top = −α/3 (virial cuántico estándar). Esto es directamente comparable con la predicción w_DE = −0,70 ± 0,05 del Cap. 6 §6.4.

In [ ]:
import numpy as np

# Organizar por g₄
by_g4 = {}
for r in results:
    by_g4.setdefault(r['g4'], []).append(r)
for g4 in by_g4:
    by_g4[g4] = sorted(by_g4[g4], key=lambda r: r['L'])

# Ajuste log-log de |ΔE| vs L
print(f'{"g₄":>8s}  {"α":>10s}  {"R²":>8s}  {"w_top=-α/3":>13s}  {"χ_H max":>10s}')
print('─' * 66)

ajustes = {}
for g4 in sorted(by_g4.keys()):
    pts = by_g4[g4]
    Ls   = np.array([p['L']  for p in pts])
    dEs  = np.array([abs(p['dE']) for p in pts])
    chis = max(max(p['chi_T3'] for p in pts), max(p['chi_R3'] for p in pts))

    valid = dEs > 0
    if valid.sum() < 3:
        alpha = np.nan; R2 = np.nan
    else:
        lL = np.log(Ls[valid]); ldE = np.log(dEs[valid])
        coef = np.polyfit(lL, ldE, 1)
        alpha = coef[0]
        ldE_pred = np.polyval(coef, lL)
        ss_res = np.sum((ldE - ldE_pred)**2)
        ss_tot = np.sum((ldE - ldE.mean())**2)
        R2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    w_top = -alpha / 3 if not np.isnan(alpha) else np.nan
    ajustes[g4] = dict(alpha=alpha, R2=R2, w_top=w_top, chi_max=chis,
                        Ls=Ls, dEs=dEs, signed_dE=np.array([p['dE'] for p in pts]))
    print(f'{g4:>8.0f}  {alpha:>+10.4f}  {R2:>8.4f}  {w_top:>+13.4f}  {chis:>10.4f}')

In [ ]:
# Análisis específico
print('\n═══ CONTROL ARMÓNICO (g₄ = 0) ═══')
alpha_arm = ajustes[0.0]['alpha']
w_arm     = ajustes[0.0]['w_top']
print(f'  α(g₄=0) medido       = {alpha_arm:+.4f}')
print(f'  α esperado (escala armónica) = −1,00')
print(f'  Diferencia |α − (−1)|        = {abs(alpha_arm + 1):.4f}')
print(f'  w_top(g₄=0) = -α/3            = {w_arm:+.4f}')
print(f'  Esperado en armónico (radiación-like) ≈ +0,333')
print(f'  → {"✓ reproduce escala armónica" if abs(alpha_arm + 1) < 0.25 else "⚠ NO reproduce"}')

print('\n═══ RÉGIMEN ANHARMÓNICO ═══')
g4_max_valid = max(g for g in ajustes.keys() if ajustes[g]['chi_max'] < 0.5 and g > 0)
alpha_max = ajustes[g4_max_valid]['alpha']
w_max     = ajustes[g4_max_valid]['w_top']
delta_alpha = alpha_max - alpha_arm
delta_w     = w_max - w_arm
print(f'  α(g₄ = {g4_max_valid:.0f}) = {alpha_max:+.4f}  (χ_H = {ajustes[g4_max_valid]["chi_max"]:.3f})')
print(f'  w_top(g₄ = {g4_max_valid:.0f}) = {w_max:+.4f}')
print(f'  Δα           = {delta_alpha:+.4f}')
print(f'  Δw_top       = {delta_w:+.4f}')

print('\n═══ COMPARACIÓN CON CAP. 6 §6.4 ═══')
print(f'  Predicción central w_DE     = −0,70 ± 0,05')
print(f'  w_top(g₄_max) sweep-L        = {w_max:+.4f}')
print(f'  Diferencia                   = {abs(w_max - (-0.70)):.4f}')
sigma_dist = abs(w_max - (-0.70)) / 0.05  # 0.05 = error sistemático del Cap. 6
print(f'  Compatibilidad con w_DE = −0,70: {sigma_dist:.2f} σ_sist')

# Veredicto cuantitativo final
print('\n═══ VEREDICTO ═══')
if abs(delta_w) < 0.05:
    veredicto = 'no hay dinámica significativa en sweep L (componente w_a despreciable)'
    color = 'NO DINÁMICA'
elif abs(delta_w) < 0.20:
    veredicto = 'dinámica moderada — componente w_a detectable'
    color = 'DINÁMICA MODERADA'
else:
    veredicto = 'dinámica fuerte — w_top cosmológico cambia sustancialmente con g₄'
    color = 'DINÁMICA FUERTE'
print(f'  {color}: {veredicto}')
print(f'  Δw_top entre g₄=0 y g₄={g4_max_valid:.0f}: {delta_w:+.3f}')

# Dirección de la evolución temporal
print(f'\n═══ DIRECCIÓN COSMOLÓGICA (DESI CPL evolutivo) ═══')
if delta_w < -0.05:
    print(f'  Δw_top < 0: w_top se vuelve MÁS NEGATIVO con g₄')
    print(f'  Como g₄·⟨φ²⟩ depende de la geometría microscópica que evoluciona con a(t),')
    print(f'  esto produce w_a < 0 en parametrización CPL (signo correcto vs DESI 2024)')
elif delta_w > 0.05:
    print(f'  Δw_top > 0: w_top se vuelve MENOS NEGATIVO con g₄')
    print(f'  Esto correspondería a w_a > 0 — dirección opuesta a DESI 2024 CPL')
else:
    print(f'  |Δw_top| pequeño: dirección no clara, w_a no determinable')

---## 7. Visualización

In [ ]:
fig, axes = new_figure(2, 2, figsize=(14, 10))
colors = plt.cm.viridis(np.linspace(0, 0.85, len(G4_VALS)))

# Panel A: |ΔE| vs L log-log
ax1 = axes[0, 0]
for g4, col in zip(sorted(by_g4.keys()), colors):
    aj = ajustes[g4]
    ax1.loglog(aj['Ls'], aj['dEs'], 'o-', color=col, ms=9, lw=2,
               label=f'g₄={g4:.0f}  α={aj["alpha"]:+.3f}')
ax1.set_xlabel('L (escala adiabática)', color=CW)
ax1.set_ylabel('|ΔE| = |E_T³ − E_R³|', color=CW)
ax1.set_title('Sweep adiabático: ΔE_top vs L\n(N fijo, V = L³)', color=CW, fontweight='bold')
ax1.legend(facecolor=BG, labelcolor=CW, fontsize=9)
ax1.grid(True, alpha=0.15, which='both')

# Panel B: α(g₄)
ax2 = axes[0, 1]
g4s_plot = np.array(sorted(by_g4.keys()))
alphas   = np.array([ajustes[g]['alpha'] for g in g4s_plot])
R2s      = np.array([ajustes[g]['R2']    for g in g4s_plot])
ax2.semilogx(np.maximum(g4s_plot, 0.1), alphas, 'o-', color=CB, ms=12, lw=2.5,
             label='α medido')
ax2.axhline(-1.0, color=CG, lw=2, ls='--', label='escala armónica (α = −1)')
ax2.axhline(+2.1, color=CR, lw=2, ls=':',  label='objetivo Cap. 6: α = +2,1 → w = −0,70')
ax2.axhline(+3.0, color='white', lw=1.5, ls=':', alpha=0.5, label='ΛCDM: α = +3 → w = −1')
for g, a, r in zip(g4s_plot, alphas, R2s):
    ax2.annotate(f'R²={r:.3f}', xy=(max(g, 0.1), a), xytext=(5, 5),
                  textcoords='offset points', fontsize=7.5, color=CGR)
ax2.set_xlabel('g₄', color=CW); ax2.set_ylabel('α (exponente ΔE ∝ L^α)', color=CW)
ax2.set_title('Sensibilidad de α al régimen anharmónico\nSweep L con N fijo', color=CW, fontweight='bold')
ax2.legend(facecolor=BG, labelcolor=CW, fontsize=8)
ax2.grid(True, alpha=0.15)

# Panel C: w_top(g₄) = −α/3 con benchmarks cosmológicos
ax3 = axes[1, 0]
w_tops = -alphas / 3
ax3.semilogx(np.maximum(g4s_plot, 0.1), w_tops, 'o-', color=CR, ms=12, lw=2.5,
             label='w_top = −α/3 (virial cuántico)')
ax3.axhline(+1/3, color=CG, lw=1.5, ls='--', alpha=0.7, label='Radiación (armónico)')
ax3.axhline(0,    color=CW, lw=1, ls='--', alpha=0.4, label='Polvo / CDM')
ax3.axhline(-2/3, color=CP, lw=1.5, ls=':', label='Frontera holográfica (−2/3)')
ax3.axhline(-0.70, color=CY, lw=2.5, label='Cap. 6 §6.4: w_DE = −0,70')
ax3.axhline(-1.0, color='white', lw=1.5, ls='--', alpha=0.5, label='ΛCDM')
ax3.set_xlabel('g₄', color=CW); ax3.set_ylabel('w_top cosmológico', color=CW)
ax3.set_title('w_top(g₄) por virial cuántico\nDirectamente comparable con DESI/Pantheon+', color=CW, fontweight='bold')
ax3.legend(facecolor=BG, labelcolor=CW, fontsize=8)
ax3.grid(True, alpha=0.15)

# Panel D: χ_H vs g₄
ax4 = axes[1, 1]
chi_max = np.array([ajustes[g]['chi_max'] for g in g4s_plot])
ax4.loglog(np.maximum(g4s_plot, 0.1), np.maximum(chi_max, 1e-6), 'o-',
           color=CO, ms=12, lw=2.5, label='χ_Hartree máx')
ax4.axhline(0.5, color=CR, lw=2, ls='--', label='límite Hartree (0,5)')
ax4.axhline(0.1, color=CG, lw=1.5, ls=':', label='Cap. 6 plateau')
ax4.set_xlabel('g₄', color=CW); ax4.set_ylabel('χ_Hartree', color=CW)
ax4.set_title('Validez del régimen Hartree', color=CW, fontweight='bold')
ax4.legend(facecolor=BG, labelcolor=CW, fontsize=9)
ax4.grid(True, alpha=0.15, which='both')

fig.suptitle(
    f'SIM 22 — Sweep adiabático L (N fijo)  →  w_top cosmológico\n'
    f'α(g₄=0)={alpha_arm:+.3f}  α(g₄_max)={alpha_max:+.3f}  '
    f'Δw_top={delta_w:+.3f}  w_top(g₄_max)={w_max:+.3f}',
    fontsize=11, fontweight='bold', color=CW)
plt.tight_layout()
plt.savefig(out_path('sim22_sweep_L.png'), dpi=150,
            bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

### Cross-validation con SIM 21Si tenés disponible el archivo `sim21_progress.json` de la corrida anterior, podés combinar ambos en un único plot que muestre los dos sweeps complementarios (β del sweep-N y α del sweep-L).

In [ ]:
# Cross-validation opcional con SIM 21
SIM21_FILE_LOC = '/content/dee_v6_topologia/sim21_progress.json'
if os.path.exists(SIM21_FILE_LOC):
    with open(SIM21_FILE_LOC) as f:
        results_s21 = json.load(f)

    # Calcular β del sweep-N para cada g₄
    by_g4_s21 = {}
    for r in results_s21:
        by_g4_s21.setdefault(r['g4'], []).append(r)

    g4_common = sorted(set(by_g4.keys()) & set(by_g4_s21.keys()))
    betas_s21 = []; alphas_s22 = []
    chi_s21 = []; chi_s22 = []

    for g4 in g4_common:
        pts21 = sorted(by_g4_s21[g4], key=lambda r: r['N'])
        Ns = np.array([p['N']  for p in pts21])
        dEs21 = np.array([abs(p['dE']) for p in pts21])
        if (dEs21 > 0).sum() >= 3:
            beta = np.polyfit(np.log(Ns), np.log(dEs21), 1)[0]
            betas_s21.append(beta)
            chi_s21.append(max(max(p['chi_T3'] for p in pts21),
                                max(p['chi_R3'] for p in pts21)))
        else:
            betas_s21.append(np.nan)
            chi_s21.append(np.nan)
        alphas_s22.append(ajustes[g4]['alpha'])
        chi_s22.append(ajustes[g4]['chi_max'])

    print('\n═══ CROSS-VALIDATION SIM 21 vs SIM 22 ═══')
    print(f'{"g₄":>8s}  {"β (sweep N)":>13s}  {"α (sweep L)":>13s}  '
          f'{"w(β)":>8s}  {"w(α)":>8s}')
    print('─' * 60)
    for i, g4 in enumerate(g4_common):
        w_b = -betas_s21[i]
        w_a = -alphas_s22[i] / 3
        print(f'{g4:>8.0f}  {betas_s21[i]:>+13.4f}  {alphas_s22[i]:>+13.4f}  '
              f'{w_b:>+8.4f}  {w_a:>+8.4f}')

    # Plot comparativo
    fig, axes = new_figure(1, 2, figsize=(14, 5))
    g4s_arr = np.array(g4_common)

    ax1 = axes[0]
    ax1.semilogx(np.maximum(g4s_arr, 0.1), betas_s21, 'o-', color=CB, ms=12, lw=2.5,
                 label='β (SIM 21, sweep N)')
    ax1.semilogx(np.maximum(g4s_arr, 0.1), alphas_s22, 's-', color=CR, ms=12, lw=2.5,
                 label='α (SIM 22, sweep L)')
    ax1.axhline(0.99, color=CG, lw=1.5, ls='--', alpha=0.7, label='Test A esperado: β=+0,99')
    ax1.axhline(-1.0, color=CY, lw=1.5, ls=':', alpha=0.7, label='Escala armónica: α=−1')
    ax1.set_xlabel('g₄', color=CW); ax1.set_ylabel('Exponente', color=CW)
    ax1.set_title('Sensibilidad de los dos exponentes a g₄', color=CW, fontweight='bold')
    ax1.legend(facecolor=BG, labelcolor=CW, fontsize=9)
    ax1.grid(True, alpha=0.15)

    ax2 = axes[1]
    w_from_beta = [-b for b in betas_s21]
    w_from_alpha = [-a/3 for a in alphas_s22]
    ax2.semilogx(np.maximum(g4s_arr, 0.1), w_from_beta,  'o-', color=CB, ms=12, lw=2.5,
                 label='w = −β (SIM 21, NO cosmológico)')
    ax2.semilogx(np.maximum(g4s_arr, 0.1), w_from_alpha, 's-', color=CR, ms=12, lw=2.5,
                 label='w = −α/3 (SIM 22, COSMOLÓGICO)')
    ax2.axhline(-0.70, color=CY, lw=2.5, label='Cap. 6 §6.4: w_DE = −0,70')
    ax2.axhline(-1.0, color='white', lw=1.5, ls='--', alpha=0.5, label='ΛCDM')
    ax2.axhline(-2/3, color=CP, lw=1.5, ls=':', label='Frontera holográfica')
    ax2.set_xlabel('g₄', color=CW); ax2.set_ylabel('w_top', color=CW)
    ax2.set_title('Comparación: convención sweep N vs cosmológica\n(la cosmológica es la que importa)',
                  color=CW, fontweight='bold')
    ax2.legend(facecolor=BG, labelcolor=CW, fontsize=8)
    ax2.grid(True, alpha=0.15)

    fig.suptitle('SIM 21 + SIM 22 — Comparación cruzada', fontsize=11, fontweight='bold', color=CW)
    plt.tight_layout()
    plt.savefig(out_path('sim21_vs_sim22.png'), dpi=150, bbox_inches='tight', facecolor='#0a0a1a')
    plt.show()
else:
    print('No se encontró sim21_progress.json — se omite cross-validation')
    print(f'Para hacerla, subir el .json a {SIM21_FILE_LOC}')

---## 8. Resumen ejecutivo consolidado

In [ ]:
print('═' * 78)
print('  SIM 22 — Sweep adiabático L con N fijo: resumen consolidado'.center(78))
print('═' * 78)
print()
print(f'{"g₄":>8s}  {"α":>10s}  {"R²":>8s}  {"w_top = -α/3":>15s}  {"χ_H max":>10s}  {"|w-(-0,70)|":>15s}')
print('─' * 78)
for g4 in sorted(by_g4.keys()):
    aj = ajustes[g4]
    diff_target = abs(aj['w_top'] - (-0.70))
    print(f'{g4:>8.0f}  {aj["alpha"]:>+10.4f}  {aj["R2"]:>8.4f}  '
          f'{aj["w_top"]:>+15.4f}  {aj["chi_max"]:>10.4f}  {diff_target:>15.4f}')
print('─' * 78)

# Conclusión cuantitativa
print()
print('CONEXIÓN CON EL DOCUMENTO DEE v6:')
print()
print(f'  • Control armónico (g₄=0): α = {alpha_arm:+.3f}')
print(f'    Esperado por teorema de escala: α = −1,0')
print(f'    {"✓ Reproduce" if abs(alpha_arm + 1) < 0.25 else "⚠ No reproduce"} (diff = {abs(alpha_arm + 1):.3f})')
print()
print(f'  • Régimen anharmónico (g₄ = {g4_max_valid:.0f}):')
print(f'    α = {alpha_max:+.3f}')
print(f'    w_top = {w_max:+.3f}')
print()
print(f'  • Comparación con predicción central w_DE = −0,70 ± 0,05:')
diff_central = abs(w_max - (-0.70))
print(f'    |Δw| = {diff_central:.3f}')
if diff_central < 0.05:
    print(f'    ✓ COINCIDE CUANTITATIVAMENTE — confirma w_DE = −0,70 desde topología anharmónica')
elif diff_central < 0.15:
    print(f'    ⚠ COMPATIBLE pero no exacta — refina la predicción central')
else:
    print(f'    ✗ DIFIERE — el sector topológico contribuye en otra dirección')
print()
print(f'  • Dinámica con g₄ (componente w_a):')
print(f'    Δw_top = {delta_w:+.3f}')
if abs(delta_w) > 0.10:
    if delta_w < 0:
        print(f'    ✓ DINÁMICA con w_a < 0 (compatible con DESI CPL evolutivo)')
    else:
        print(f'    ⚠ DINÁMICA con w_a > 0 (dirección opuesta a DESI)')
elif abs(delta_w) > 0.03:
    print(f'    ~ Componente w_a sub-líder detectable')
else:
    print(f'    ○ Sin dinámica significativa en este rango de g₄')

# Archivos
print()
print('Archivos generados:')
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f'  {f}  ({sz/1024:.1f} KB)')

In [ ]:
# Empaquetar
import subprocess
subprocess.run(['zip', '-q', '-r', os.path.join(OUT_DIR, 'sim22_resultados.zip'), '.'], cwd=OUT_DIR)
print(f'\n→ {OUT_DIR}/sim22_resultados.zip')

---## Qué hacer con los resultados1. Descargar `sim22_resultados.zip` (panel izquierdo de Colab)2. Pasarme:   - El `.png` principal (`sim22_sweep_L.png`)   - Si está disponible, el `.png` de cross-validation (`sim21_vs_sim22.png`)   - La salida de texto del resumen consolidado3. Con eso cerramos la subsección del Cap. 6 §6.3 y actualizamos:   - Cap. 6 §6.4 (predicción central refinada con dirección de w_a)   - Cap. 7 §7.3 (compromiso de falsabilidad → predicción positiva refinada)   - Apéndice A (filas nuevas con α, w_top sweep-L)**Si algo falla:**- Para N=1331 cada Hartree toma ~15 s; si querés acelerar, podés bajar a N_SIDE=9 (N=729): 4 × más rápido. La precisión cae algo pero el efecto cualitativo es el mismo.- Si Hartree no converge en algún g₄ alto: aumentar `HARTREE_MAX` de 25 a 40.- Si querés explorar más g₄ (entre 5000 y 10000), agregar `2500.0, 3500.0` a `G4_VALS`.